# 🪖 Notebook 01 — Synthetic IMU Dataset Generation

**Project:** Smart Airbag Helmet — Pre-Impact Rider State Classification  
**Goal:** Generate a realistic time-series IMU dataset since we don't have hardware yet.  
**Output:** `data/synthetic/helmet_imu_raw.csv`

---

## Why time-series instead of independent rows?

Real sensor data is sequential. A crash isn't one row — it's a **pattern** that unfolds over time:  
`Normal → Sudden Brake → Impact Spike → Post-crash chaos`

By generating sessions (sequences), we're building a dataset that's compatible with:  
- Sliding Window → Random Forest  
- LSTM / Transformer (later)  
- Real-time edge prediction on ESP32  

---

## Classes

| Label | ID | Real-world meaning |
|---|---|---|
| 🟢 Normal Riding | 0 | Stable, no event |
| 🟡 Pothole | 1 | Short vertical az spike |
| 🟠 Sudden Brake | 2 | Strong ax deceleration + pitch |
| 🔴 Crash / Fall | 3 | Multi-axis spike, sustained chaos |

In [ ]:
import sys
import os

# Add project root to path so we can import from src/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

# Import our generator
from src.data_generator import generate_dataset, generate_session, LABEL_MAP

print('All imports OK')
print(f'NumPy  : {np.__version__}')
print(f'Pandas : {pd.__version__}')

## Step 1 — Understand what MPU6050 gives us

The MPU6050 is a 6-DOF IMU (Inertial Measurement Unit):

```
Accelerometer (m/s²)       Gyroscope (°/s)
─────────────────────       ──────────────────
ax  ← left/right           gx  ← roll rate
ay  ← forward/backward     gy  ← pitch rate
az  ← up/down              gz  ← yaw rate
```

At rest: `az ≈ 9.81 m/s²` (gravity).  
During crash: `all axes spike wildly`, gravity vector is lost.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1: Quick single-session preview (one ride)
# ─────────────────────────────────────────────────────────────────

rng = np.random.default_rng(42)
sample_session = generate_session(session_id=0, session_len=100, rng=rng)

print('Single session shape:', sample_session.shape)
print('\nFirst 10 rows:')
display(sample_session.head(10))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 2: Visualize a single session — see the patterns
# ─────────────────────────────────────────────────────────────────

COLOR_MAP = {0: '#2ecc71', 1: '#f1c40f', 2: '#e67e22', 3: '#e74c3c'}
LABEL_NAMES = {0: 'Normal', 1: 'Pothole', 2: 'SuddenBrake', 3: 'Crash'}

def plot_session(df: pd.DataFrame, title: str = 'Riding Session'):
    """Plot all 6 IMU axes for one session, colored by label."""
    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor('#0d0d0d')
    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

    axes_names = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
    y_labels   = ['ax (m/s²)', 'ay (m/s²)', 'az (m/s²)', 'gx (°/s)', 'gy (°/s)', 'gz (°/s)']

    for idx, (col, ylabel) in enumerate(zip(axes_names, y_labels)):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        ax.set_facecolor('#1a1a2e')
        ax.tick_params(colors='#aaaaaa', labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor('#333355')

        # Plot segments colored by label
        t = df['timestamp'].values
        y = df[col].values
        labels = df['label'].values

        for i in range(len(t) - 1):
            ax.plot([t[i], t[i+1]], [y[i], y[i+1]],
                    color=COLOR_MAP[labels[i]], linewidth=1.5, alpha=0.9)

        ax.set_title(ylabel, color='#e0e0e0', fontsize=10, pad=5)
        ax.set_xlabel('Timestamp', color='#888888', fontsize=8)
        ax.grid(True, color='#222244', linewidth=0.5)

    # Legend
    legend_elements = [Patch(facecolor=COLOR_MAP[k], label=v) for k, v in LABEL_NAMES.items()]
    fig.legend(handles=legend_elements, loc='upper center', ncol=4,
               facecolor='#1a1a2e', edgecolor='#444466',
               labelcolor='#e0e0e0', fontsize=9,
               bbox_to_anchor=(0.5, 0.97))

    fig.suptitle(title, color='#ffffff', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


plot_session(sample_session, title='Sample Riding Session — All 6 IMU Channels')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 3: Visualize each class in isolation
# ─────────────────────────────────────────────────────────────────
from src.data_generator import _normal, _pothole, _sudden_brake, _crash

rng = np.random.default_rng(99)
n_samples = 60

gen_map = {
    '🟢 Normal':        _normal(n_samples, rng),
    '🟡 Pothole':       _pothole(n_samples, rng),
    '🟠 Sudden Brake':  _sudden_brake(n_samples, rng),
    '🔴 Crash':         _crash(n_samples, rng),
}

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
fig.patch.set_facecolor('#0d0d0d')
colors_per_class = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

for row_idx, (class_name, signals) in enumerate(gen_map.items()):
    color = colors_per_class[row_idx]
    t = np.arange(n_samples)

    # Accel magnitude
    ax0 = axes[row_idx, 0]
    ax0.set_facecolor('#1a1a2e')
    accel_mag = np.sqrt(signals['ax']**2 + signals['ay']**2 + signals['az']**2)
    ax0.plot(t, accel_mag, color=color, linewidth=2)
    ax0.set_title(f'{class_name} — Accel Magnitude', color='#e0e0e0', fontsize=9)
    ax0.tick_params(colors='#aaaaaa')
    ax0.set_facecolor('#1a1a2e')
    ax0.grid(True, color='#222244', linewidth=0.5)
    for spine in ax0.spines.values(): spine.set_edgecolor('#333355')

    # Gyro magnitude
    ax1 = axes[row_idx, 1]
    gyro_mag = np.sqrt(signals['gx']**2 + signals['gy']**2 + signals['gz']**2)
    ax1.plot(t, gyro_mag, color=color, linewidth=2, linestyle='--')
    ax1.set_title(f'{class_name} — Gyro Magnitude', color='#e0e0e0', fontsize=9)
    ax1.tick_params(colors='#aaaaaa')
    ax1.set_facecolor('#1a1a2e')
    ax1.grid(True, color='#222244', linewidth=0.5)
    for spine in ax1.spines.values(): spine.set_edgecolor('#333355')

fig.suptitle('Class Signal Profiles — Accel & Gyro Magnitude', color='white', fontsize=13)
plt.tight_layout()
plt.show()

print('\nKey observations:')
print('  Normal    — low, flat accel (~9.81), near-zero gyro')
print('  Pothole   — brief az spike, small gyro burst')
print('  Brake     — sustained ax decel, elevated gy (pitch forward)')
print('  Crash     — MASSIVE spike in all axes, chaotic post-crash')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 4: Generate full dataset
# ─────────────────────────────────────────────────────────────────

import os

SAVE_PATH = os.path.join('..', 'data', 'synthetic', 'helmet_imu_raw.csv')

dataset = generate_dataset(
    n_sessions  = 500,
    session_len = 100,
    seed        = 42,
    save_path   = SAVE_PATH,
    verbose     = True
)

print('\nDataset dtypes:')
print(dataset.dtypes)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 5: Quick sanity checks
# ─────────────────────────────────────────────────────────────────

print('=== DATASET OVERVIEW ===')
print(f'Shape          : {dataset.shape}')
print(f'Total sessions : {dataset.session_id.nunique()}')
print(f'Missing values : {dataset.isnull().sum().sum()}')

print('\n=== DESCRIPTIVE STATS (sensor columns) ===')
display(dataset[['ax','ay','az','gx','gy','gz']].describe().round(3))

print('\n=== LABEL DISTRIBUTION ===')
display(dataset['label_name'].value_counts().to_frame('count'))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 6: Label distribution pie chart
# ─────────────────────────────────────────────────────────────────

counts  = dataset['label_name'].value_counts()
colors  = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
ordered = ['Normal', 'Pothole', 'SuddenBrake', 'Crash']
vals    = [counts.get(k, 0) for k in ordered]

fig, (ax_pie, ax_bar) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0d0d0d')

# Pie
ax_pie.set_facecolor('#0d0d0d')
wedges, texts, autotexts = ax_pie.pie(
    vals, labels=ordered, colors=colors,
    autopct='%1.1f%%', startangle=140,
    textprops={'color': '#e0e0e0', 'fontsize': 10},
    wedgeprops={'linewidth': 1.5, 'edgecolor': '#0d0d0d'}
)
for at in autotexts: at.set_color('white')
ax_pie.set_title('Label Distribution', color='white', fontsize=12)

# Bar
ax_bar.set_facecolor('#1a1a2e')
bars = ax_bar.bar(ordered, vals, color=colors, edgecolor='#0d0d0d', linewidth=1.5)
for bar, val in zip(bars, vals):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
               f'{val:,}', ha='center', color='white', fontsize=9)
ax_bar.set_facecolor('#1a1a2e')
ax_bar.tick_params(colors='#aaaaaa')
ax_bar.set_ylabel('Count', color='#aaaaaa')
ax_bar.set_title('Samples per Class', color='white', fontsize=12)
ax_bar.grid(True, axis='y', color='#222244', linewidth=0.5)
for spine in ax_bar.spines.values(): spine.set_edgecolor('#333355')

fig.suptitle('Class Balance — helmet_imu_raw.csv', color='white', fontsize=13)
plt.tight_layout()
plt.show()

print('\nNote: Normal dominates because most riding is uneventful.')
print('We will handle class imbalance during model training (class_weight="balanced").')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 7: View a session with a crash in it
# ─────────────────────────────────────────────────────────────────

# Find sessions that contain crashes
crash_sessions = dataset[dataset['label'] == 3]['session_id'].unique()
print(f'Sessions containing crashes: {len(crash_sessions)}')

if len(crash_sessions) > 0:
    crash_ex = dataset[dataset['session_id'] == crash_sessions[0]]
    plot_session(crash_ex, title=f'Session {crash_sessions[0]} — Contains Crash Event')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 8: Per-feature box plots by class
# ─────────────────────────────────────────────────────────────────

import matplotlib.patches as mpatches

features     = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
class_colors = {0: '#2ecc71', 1: '#f1c40f', 2: '#e67e22', 3: '#e74c3c'}
class_names  = {0: 'Normal', 1: 'Pothole', 2: 'SuddenBrake', 3: 'Crash'}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor('#0d0d0d')
axes_flat = axes.flatten()

for i, feat in enumerate(features):
    ax = axes_flat[i]
    ax.set_facecolor('#1a1a2e')

    data_by_class = [dataset[dataset['label'] == lbl][feat].values for lbl in range(4)]

    bp = ax.boxplot(data_by_class, patch_artist=True, notch=False,
                    whiskerprops=dict(color='#888888'),
                    capprops=dict(color='#888888'),
                    medianprops=dict(color='white', linewidth=2),
                    flierprops=dict(marker='o', markerfacecolor='#ff6b6b', markersize=2, alpha=0.4))

    for patch, lbl in zip(bp['boxes'], range(4)):
        patch.set_facecolor(class_colors[lbl])
        patch.set_alpha(0.7)

    ax.set_xticklabels([class_names[j] for j in range(4)], color='#aaaaaa', fontsize=8, rotation=15)
    ax.set_title(feat, color='#e0e0e0', fontsize=11)
    ax.tick_params(colors='#aaaaaa')
    ax.grid(True, axis='y', color='#222244', linewidth=0.5)
    for spine in ax.spines.values(): spine.set_edgecolor('#333355')

fig.suptitle('Feature Distributions by Class', color='white', fontsize=14)
plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('  ax: SuddenBrake has strongly negative values (deceleration)')
print('  az: Pothole spikes positive, Crash loses gravity')
print('  gx/gy/gz: Crash has enormous outliers (±100-300°/s)')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 9: Correlation heatmap
# ─────────────────────────────────────────────────────────────────

import matplotlib.colors as mcolors

corr = dataset[['ax','ay','az','gx','gy','gz','label']].corr()

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#1a1a2e')

cmap = plt.cm.RdYlGn
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')

cols = corr.columns.tolist()
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, color='#e0e0e0')
ax.set_yticklabels(cols, color='#e0e0e0')

for i in range(len(cols)):
    for j in range(len(cols)):
        val = corr.values[i, j]
        text_color = 'black' if abs(val) < 0.6 else 'white'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color=text_color, fontsize=9)

plt.colorbar(im, ax=ax, label='Correlation')
ax.set_title('Feature Correlation Matrix', color='white', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 10: Summary — what we built
# ─────────────────────────────────────────────────────────────────

print('=' * 55)
print('  DATASET GENERATION COMPLETE')
print('=' * 55)
print(f'  File     : data/synthetic/helmet_imu_raw.csv')
print(f'  Shape    : {dataset.shape}')
print(f'  Sessions : {dataset.session_id.nunique()}')
print(f'  Columns  : {list(dataset.columns)}')
print()
print('  Next: notebooks/02_eda.ipynb')
print('        → Sliding Window + Feature Engineering')
print('=' * 55)

---

## ✅ What we built

| Item | Detail |
|---|---|
| Dataset | 500 sessions × ~100 timesteps ≈ 50,000 rows |
| Classes | Normal, Pothole, SuddenBrake, Crash |
| Realism | Physics-based distributions, not random noise |
| Structure | Time-series (sequential, not IID rows) |
| Format | CSV with session_id + timestamp → sliding window ready |

---

## 🔜 Next Notebook: `02_eda.ipynb`

We'll apply a **sliding window** over sessions and extract features:
- Mean, Std, Max, Min per axis
- Accel magnitude, Gyro magnitude
- Jerk (derivative of accel)
- Tilt angle

Then feed those into **Random Forest** and compare models.